In [1]:
'ciao'

'ciao'

In [2]:
import os
import os.path as osp
if osp.split(os.getcwd())[-1] == 'notebooks':
    os.chdir('..')

os.getcwd()

'e:\\Edoardo\\Education\\SportDataCampus\\ScoutingAgent'

In [3]:
from wyscout import get_match_events, get_match_details

# Start

In [4]:
events = get_match_events(5718124)['events']

In [5]:
def get_possession_events(events: list[dict], possession_id: int) -> list[dict]:
    """
    Extracts possession events from a list of Wyscout events.

    Args:
        events (list[dict]): A list of dictionaries containing Wyscout event data.
    """

    possession_events = list(
        filter(
            lambda event: event.get('possession') is not None and event['possession'].get('id') == 101,
            events
        )
    )
    return possession_events

In [6]:
def get_match_possession(events: list[dict]) -> list[dict]:

    """
    Returns a list of all unique possessions in the match as lists of event dicts (ordered as in events).

    Args:
        events (list[dict]): A list of dictionaries containing Wyscout event data for the match.

    Returns:
        list[dict]: A list where each item is a dict:
            {
                "possession_id": int,
                "events": list[dict]
            }
    """
    from collections import defaultdict

    # Group events by possession id
    possessions = defaultdict(list)
    for event in events:
        pid = None
        if event.get('possession') and event['possession'].get('id') is not None:
            pid = event['possession']['id']
        if pid is not None:
            possessions[pid].append(event)

    # Result: list of dicts with possession_id and events
    return [
        {
            "possession_id": pid,
            "events": possession_events
        }
        for pid, possession_events in sorted(possessions.items())
    ]

In [7]:
possessions = get_match_possession(events)

## Possession analysis

In [8]:
pid = possessions[0]['possession_id']
possession_events = possessions[0]['events']
match_info = get_match_details(5718124, details=['teams'])

In [9]:
from possession_analyzer import analyze_possession

possession_analysis = analyze_possession(possession_events, match_info, events)

In [10]:
match_info['teamsData']['692']['team'].keys()

dict_keys(['wyId', 'name', 'officialName', 'city', 'area', 'type', 'category', 'gender', 'children', 'imageDataURL'])

In [11]:
possession_analysis

{'possession_id': 2927832885,
 'num_events': 4,
 'team_in_possession': 675,
 'team_in_possession_name': 'Real Madrid',
 'opponent_team_name': 'Celta de Vigo',
 'opponent_team_id': 692,
 'duration': 1.48391,
 'pass_count': 0,
 'avg_pass_speed': 0.0,
 'total_x_advancement': -3.0,
 'time_in_thirds': {'defensive': 100.0, 'middle': 0.0, 'attacking': 0.0},
 'ball_circulation_count': 0,
 'match_state': {'home_score': 0, 'away_score': 0, 'leading_team': None},
 'players_involved': [{'id': 276979, 'name': 'F. Mendy', 'team_id': 675},
  {'id': 569212, 'name': 'Ferrán Jutglà', 'team_id': 692}],
 'temporal_moment': {'period': '1H', 'matchTimestamp': '00:00:07.268'},
 'preceding_events': [{'id': 2927833236,
   'matchId': 5718124,
   'matchPeriod': '1H',
   'minute': 0,
   'second': 4,
   'matchTimestamp': '00:00:04.866',
   'videoTimestamp': '5.866453',
   'relatedEventId': 2927833238,
   'type': {'primary': 'pass',
    'secondary': ['forward_pass',
     'long_pass',
     'loss',
     'pass_to_fina

## Vertex AI

In [12]:
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_APPLICATION_CREDENTIALS"]

'.secrets/vertexai-wyscout.json'

In [13]:
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(
    project=os.environ["GCP_PROJECT_ID"],
    location=os.environ["VERTEX_LOCATION"],
)
model = GenerativeModel(os.environ["VERTEX_MODEL_NAME"])

e:\Edoardo\Education\SportDataCampus\ScoutingAgent\.venv\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [14]:
resp = model.generate_content("Rispondi in una riga: che modello sei?")
resp

candidates {
  content {
    role: "model"
    parts {
      text: "Sono un modello linguistico di grandi dimensioni, addestrato da Google."
    }
  }
  finish_reason: STOP
  avg_logprobs: -0.00032898519809047383
}
usage_metadata {
  prompt_token_count: 12
  candidates_token_count: 15
  total_token_count: 27
  prompt_tokens_details {
    modality: TEXT
    token_count: 12
  }
  candidates_tokens_details {
    modality: TEXT
    token_count: 15
  }
}
model_version: "gemini-2.5-flash-lite"
create_time {
  seconds: 1774187237
  nanos: 202672000
}
response_id: "5fK_abCvDKC1mPUPl_mCgA4"

In [15]:
from possession_description import render_general_possession_prompt

In [16]:
possession_events

[{'id': 2927832885,
  'matchId': 5718124,
  'matchPeriod': '1H',
  'minute': 0,
  'second': 7,
  'matchTimestamp': '00:00:07.268',
  'videoTimestamp': '8.268542',
  'relatedEventId': 2927832886,
  'type': {'primary': 'duel',
   'secondary': ['loose_ball_duel', 'recovery', 'counterpressing_recovery']},
  'location': {'x': 30, 'y': 3},
  'team': {'id': 675, 'name': 'Real Madrid', 'formation': '5-4-1'},
  'opponentTeam': {'id': 692, 'name': 'Celta de Vigo', 'formation': '3-4-3'},
  'player': {'id': 276979, 'name': 'F. Mendy', 'position': 'LB5'},
  'pass': None,
  'shot': None,
  'groundDuel': None,
  'aerialDuel': None,
  'infraction': None,
  'carry': None,
  'possession': {'id': 2927832885,
   'duration': '1.48391',
   'types': [],
   'eventsNumber': 4,
   'eventIndex': 0,
   'startLocation': {'x': 30, 'y': 3},
   'endLocation': {'x': 27, 'y': 0},
   'team': {'id': 675, 'name': 'Real Madrid', 'formation': '5-4-1'},
   'attack': None}},
 {'id': 2927833238,
  'matchId': 5718124,
  'matchP

In [17]:
prompt = render_general_possession_prompt(possession_analysis)
print(prompt)

UndefinedError: 'output_format_constraints' is undefined

In [ ]:
import os
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig

from possession_analyzer import analyze_possession  # o come lo importi tu
from possession_description import generate_possession_description

vertexai.init(
    project=os.environ["GCP_PROJECT_ID"],
    location=os.environ["VERTEX_LOCATION"],
)

model = GenerativeModel(os.environ["VERTEX_MODEL_NAME"])

# possession_events: lista eventi Wyscout ordinati per quel possesso
# match_info: output get_match_details (con squadre, ecc.)
# all_match_events: tutti gli eventi partita se vuoi preceding + match_state
analysis = analyze_possession(
    possession_events,
    match_info,
    all_match_events=events,
)

text = generate_possession_description(
    model,
    analysis,
    generation_config=GenerationConfig(
        temperature=0.4,
        max_output_tokens=2048,
    ),
)
print(text)

e:\Edoardo\Education\SportDataCampus\ScoutingAgent\.venv\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


SECTION 1 - GENERAL POSSESSION ANALYSIS:

This possession sequence for Real Madrid was extremely brief and initiated immediately following a Celta de Vigo attacking sequence that culminated in a turnover. The preceding events show Celta de Vigo building play from their defensive third, with Ilaix Moriba making a short pass to Marcos Alonso in the middle third. Alonso then executed a progressive, high, long pass into the attacking third, aimed at Ferrán Jutglà. However, this pass was intercepted or lost, leading directly into Real Madrid's possession.

The overall flow of play was characterized by an immediate defensive action and a rapid loss of possession. Real Madrid's possession began with a duel in the defensive third, specifically in the left zone. F. Mendy was involved in a loose ball duel, described as a counter-pressing recovery, indicating an immediate attempt to regain possession after losing it. This was quickly followed by a duel from Celta de Vigo's Ferrán Jutglà in the sa

In [18]:
from possession_description import (
    render_player_section_possession_prompt,
    build_target_player_events_text,
)

# analysis = analyze_possession(possession_events, match_info, all_match_events=events)

general_text = "..."  # output reale Section 1, oppure "PLACEHOLDER general analysis"
pid = 276979  # Wyscout player id

print(build_target_player_events_text(analysis, pid))  # JSON eventi del giocatore
print(
    render_player_section_possession_prompt(
        analysis,
        general_description=general_text,
        target_player_id=pid,
        # target_player_name="F. Mendy",  # opzionale
    )
)

NameError: name 'analysis' is not defined